# CEP: Gradient Boosting Trees

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [2]:
import pandas as pd
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm
import catboost as cb

from models.utilities import dw
from data_helpers.preprocessors.scalers import denormalize_feats, normalize_feats, normalize_other

## Keep relevant columns and prepare train-test loader

In [3]:
model_name = 'gbt'
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft').reset_index(drop=True)
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Prepare categorical features

In [4]:
cat_feats = ['feedback_setting', 'price_rule']
time_aggregated_dataset = pd.get_dummies(time_aggregated_dataset, columns=cat_feats)
dummy_vars_columns = list(filter(lambda x: 'feedback_setting_' in x or 'price_rule_' in x ,time_aggregated_dataset.columns))
target_col = 'ce_round'

## Keep relevant columns and prepare train-test loader

In [5]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
feature_cols = [col for col in time_aggregated_dataset.columns.values if "change" not in col and "running_" in col]
time_columns =  ['n_unique_deals_round'] + ['round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)

## Select Hyper-parameter Grid for GBT

In [6]:
min_quantile = 0.35
max_quantile = 1 - min_quantile
hyperparameter_grid = dict(n_estimators=[1, 2, 3],
                           depth=[1,2,3],
                           learning_rate=[0.001, 0.005, 0.01],  
                           max_leaves=[10, 20, 30, 40, 50, 70],
                          )

## Fit and evaluate model

In [7]:
np.random.seed(1)
all_results = []
regression_res = []

for i in tqdm(range(pdl.max_samples)):
    # Get data set split
    train_df, test_df = pdl.get_sample_split_dataset(i)
    sub_train_df = train_df.copy()

    # Initialize Model
    gbt_regressor = cb.CatBoostRegressor(boost_from_average=True, 
                                     loss_function='Quantile',
                                     grow_policy='Lossguide',
                                     task_type="CPU",
                                    )

    # Normalize all numerical features with IQR and Median standardization.
    X = sub_train_df[feature_cols].values
    X_norm, X_median, X_iqr = normalize_feats(X, min_quantile, max_quantile)
    y = sub_train_df[target_col].values[:, np.newaxis]
    
    # Normalize target CEP with same median and IQR values.
    y_norm = normalize_other(y,  X_median, X_iqr)

    # Add time and categorical features to normalized data.
    X_other = sub_train_df[dummy_vars_columns+time_columns].values
    X_all = np.concatenate([X_norm, X_other], axis=1)

    # Perform cross validation on train set and pick best hyper-parameters and model
    best_model_params = gbt_regressor.grid_search(hyperparameter_grid,
                                         X_all,
                                         y=y_norm,
                                         cv=5,
                                         partition_random_seed=0,
                                         calc_cv_statistics=True,
                                         search_by_train_test_split=True,
                                         refit=True,
                                         shuffle=True,
                                         stratified=None,
                                         train_size=0.8,
                                         verbose=False,
                                         plot=False,
                                         log_cout=dw,
                                         log_cerr=sys.stderr,
                                  )
    
    best_model = gbt_regressor

    # Retrieve feature importances of best models
    assert len(feature_cols + dummy_vars_columns+time_columns) == len(best_model.feature_importances_)
    feat_importance_df = pd.Series(dict(zip(
    feature_cols + dummy_vars_columns+time_columns, best_model.feature_importances_.tolist()))).to_frame().T
    feat_importance_df['sample_id'] = i
    
    # Persist resulting model parameters and feature importance
    best_params_df = pd.Series(best_model_params['params']).to_frame().T
    a = pd.concat([feat_importance_df, best_params_df], axis=1, ignore_index=False)
    regression_res.append(a)

    for rd in rounds:  
        for n_deal_price in n_deal_prices:
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query).copy()

            # Normalize test numerical features with observed median and IQR values
            X_test_price_feats = sub_test_set[feature_cols].values
            X_test_norm, X_test_median, X_test_iqr = normalize_feats(X_test_price_feats, min_quantile, max_quantile)

            # Concatenate with time and categorical features
            X_other_test = sub_test_set[dummy_vars_columns+time_columns].values
            X_all_test = np.concatenate([X_test_norm, X_other_test], axis=1)
            prediction = best_model.predict(X_all_test)            
            scaled_pred = denormalize_feats(prediction[:, np.newaxis], X_test_median, X_test_iqr)    
            
            # Calculate APE between predictions and targets
            test_targets = sub_test_set[target_col].values[:, np.newaxis]
            result_test_df = sub_test_set[key_columns].copy()
            result_test_df.loc[:, 'sample_id'] = i
            result_test_df.loc[:, 'ce_ape'] = (np.abs(scaled_pred - test_targets)/test_targets)
            all_results.append(result_test_df)

  0%|          | 0/50 [00:00<?, ?it/s]

## Combine Results

In [8]:
all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name
regression_data_df = pd.concat(regression_res, axis=0, ignore_index = True)
regression_data_df['model'] = model_name

## Persist Results

In [9]:
all_results_df.to_feather('../../../data/results/ce_price/'+model_name+'.ft')
regression_data_df.to_feather('../../../data/results/ce_price/'+model_name+'_data.ft')